In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!cp /content/drive/MyDrive/modules_work/COMP34612/comp34612.zip /content/

<a href="https://colab.research.google.com/github/lpankhurst/t2-cwk/blob/main/comp34612_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

TODO:

More hyperparams.


# Install and Import

In [ ]:
import zipfile, os, random, gc, math
from IPython.display import Javascript
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 15)
pd.options.display.float_format = '{:.6f}'.format

In [ ]:
extract_path = "."
zip_filename = "comp34612.zip"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_filename, "r") as zip_ref:
    zip_ref.extractall(extract_path)

In [ ]:
!pip install xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 10.8 MB/s eta 0:00:00


In [ ]:
##  Note that xlsxwriter is essential for these imports to work (and therefore the simulation entirely).

from engine import Engine
from gui import GUI

Please import the necessary packages here or use `!pip install *package_name*` if they are not already installed.

# Base Leader and Example Leader

## Base Leader

This is the base class for all leader subclasses.

In [265]:
class Leader:
    _subclass_registry = {}

    def __init__(self, name, engine):
        self.name = name
        self.engine = engine

    @classmethod
    def cleanup_old_subclasses(cls):
        """
        A function to remove old subclasses before defining new ones.
        """
        existing_subclasses = list(cls.__subclasses__())

        for subclass in existing_subclasses:
            subclass_name = subclass.__name__
            if subclass_name in cls._subclass_registry:
                del cls._subclass_registry[subclass_name]
                del subclass
        gc.collect()

    @classmethod
    def update_subclass_registry(cls):
        """
        A function to update registry after cleaning up old subclasses.
        """
        cls.cleanup_old_subclasses()
        cls._subclass_registry = {subclass.__name__: subclass for subclass in cls.__subclasses__()}

    def new_price(self, date):
        """
        A function for setting the new price of each day.
        :param date: date of the day to be updated
        :return: (float) price for the day
        """
        pass

    def get_price_from_date(self, date):
        """
        A function for getting the price set on a date.
        :param date: (int) date to get the price from
        :return: a tuple (leader_price, follower_price)
        """
        return self.engine.exposed_get_price(date)


    def start_simulation(self):
        """
        A function runs at the beginning of the simulation.
        """
        pass

    def end_simulation(self):
        """
        A function runs at the beginning of the simulation.
        """
        pass

## Example Leader

This is a simple example leader subclass.

In [182]:
class SimpleLeader(Leader):
    def __init__(self, name, engine):
        super().__init__(name, engine)

    def new_price(self, date: int):
        return 1.5 + random.random() * 0.1

## Generic functions

In [183]:
def load_xls_to_df() -> dict[str, pd.DataFrame]:

  ##  Converts Excel file to dictionary of Pandas dataframes.

  dfs = pd.read_excel('data.xlsx', sheet_name = None, index_col = "Date")

  return dfs

In [184]:
def loaded_dfs_setup(dfs: dict[str, pd.DataFrame]) -> dict[str, pd.DataFrame]:

  ##  Sets up extra dataframe columns; namely profit and n-order derivatives.

  ##  S = 100 − 5uL + 3uF
  ##  P = (u_L - u_C) * S

  dfs_new = {}

  for follower, df_follower in dfs.items():

    dfs_new[follower] = df_follower.copy()

    dfs_new[follower]["Leader's Price dL/dD"] = dfs_new[follower]["Leader's Price"].diff().fillna(0)

    dfs_new[follower]["Leader's Price d2L/dD2"] = dfs_new[follower]["Leader's Price dL/dD"].diff().fillna(0)

    dfs_new[follower]["Profit"] = (df_follower["Leader's Price"] - df_follower["Cost"]) * (100 - 5 * df_follower["Leader's Price"] + 3 * df_follower["Follower's Price"])

    dfs_new[follower]["Profit dP/dL"] = dfs_new[follower]["Profit"].diff().fillna(0)

    dfs_new[follower]["Profit d2P/dL2"] = dfs_new[follower]["Profit dP/dL"].diff().fillna(0)

    dfs_new[follower]["Profit d3P/dL3"] = dfs_new[follower]["Profit d2P/dL2"].diff().fillna(0)

  return dfs_new

In [185]:
dfs_loaded = load_xls_to_df()

##  print(dfs_loaded)

dfs_primed = loaded_dfs_setup(dfs_loaded)

print(dfs_primed["Follower_Mk3"].head(2))

      Leader's Price  Follower's Price  Cost  Leader's Price dL/dD  \
Date                                                                 
1           1.780782          1.278121     1              0.000000   
2           1.730777          1.437587     1             -0.050005   

      Leader's Price d2L/dD2    Profit  Profit dP/dL  Profit d2P/dL2  \
Date                                                                   
1                   0.000000 74.120001      0.000000        0.000000   
2                  -0.050005 69.905300     -4.214702       -4.214702   

      Profit d3P/dL3  
Date                  
1           0.000000  
2          -4.214702  


In [186]:
def extract_initial_stats(df: pd.DataFrame,
                          initial_boost: int) -> dict[str, float]:

  ##  Derives data to write to date 101.

  stats = {}

  res = df.sort_values(by = "Profit").iloc[74] ##.iloc[0]

  print(df["Leader's Price"].quantile(0.75))

  # stats["Leader's Price"] = res["Leader's Price"]

  stats["Leader's Price"] = res["Cost"]

  stats["Follower's Price"] = res["Leader's Price"]

  stats["Cost"] = res["Cost"]

  stats["Leader's Price dL/dD"] = abs(res["Leader's Price dL/dD"]) * initial_boost

  stats["Leader's Price d2L/dD2"] = 0

  stats["Profit"] = 0

  stats["Profit dP/dL"] = 0

  stats["Profit d2P/dL2"] = 0

  stats["Profit d3P/dL3"] = 0

  stats_floats = {}

  for k, v in stats.items():

    stats_floats[k] = float(v)

  return stats_floats

In [187]:
def update_history(df: pd.DataFrame,
                   date: int,
                   cost: float,
                   prices: tuple[float, float],
                   row_stats: dict[str, float]) -> pd.DataFrame:

  ##  Updates 30-day dataframe with full row of data, based on latest leaders & follower prices.

  print("update_history()")

  new_row = {}

  new_row["Cost"] = cost

  new_row["Leader's Price"] = prices[0]

  new_row["Follower's Price"] = prices[1]

  print("cost, LP, FP:")
  print((cost, prices[0], prices[1]))

  new_row["Profit"] = (new_row["Leader's Price"] - new_row["Cost"]) * (100 - 5 * new_row["Leader's Price"] + 3 * new_row["Follower's Price"])

  print(f"new row in progress for date {date}", new_row)

  if (date == 101):

    for field in ["Leader's Price dL/dD",
                  "Leader's Price d2L/dD2",
                  "Profit dP/dL",
                  "Profit d2P/dL2",
                  "Profit d3P/dL3"]:

      new_row[field] = row_stats[field]

  else:

    new_row["Leader's Price dL/dD"] = new_row["Leader's Price"] - df.loc[date - 1]["Leader's Price"]

    new_row["Leader's Price d2L/dD2"] = new_row["Leader's Price dL/dD"] - df.loc[date - 1]["Leader's Price dL/dD"]

    print("new row LP done", new_row)

    new_row["Profit dP/dL"] = new_row["Profit"] - df.loc[date - 1]["Profit"]

    new_row["Profit d2P/dL2"] = new_row["Profit dP/dL"] - df.loc[date - 1]["Profit dP/dL"]

    new_row["Profit d3P/dL3"] = new_row["Profit d2P/dL2"] - df.loc[date - 1]["Profit d2P/dL2"]

    print("new row profit done", new_row)

  new_row_typed = {}

  for k, v in new_row.items():

    # print(k, v)

    new_row_typed[k] = float(v)

  print("new_row_typed", new_row_typed)

  return new_row_typed

<a name="your-leader"></a>
# Your Leader

Notes:

*  Only classes that inherit from Leader will show in GUI (cannot make another intermediate base class)
*  `unsupported operand type(s) for *: 'float' and 'NoneType'` occurs when a price was not returned by `new_price()`


In [188]:
group_num = 23

assert isinstance(group_num, int), f"Expected an integer for group_num, but got {type(group_num).__name__}"

In [267]:
class Optimum_chaser_generic(Leader):

  ##  Note that this is also a base class;
  ##  you should only instantiate instances of the MK* subclasses.

  def __init__(self, name, engine):

    ##  Load data, setup DFs.

    dfs_loaded = pd.read_excel('data.xlsx', sheet_name = None, index_col = "Date")

    self.dfs_primed = loaded_dfs_setup(dfs_loaded)

    ##  Setup new DF for new 30 days.

    columns_set = list(self.dfs_primed[self.target])

    self.df_30 = pd.DataFrame(columns = columns_set)

    self.df_orientation = pd.DataFrame(columns = ["Leader's Price",
                                                  "Profit",
                                                  "Big dP/dL",
                                                  "Tiny dP/dL"])

    self.df_grads = pd.DataFrame(columns = [
        "Leader's Price",
        "dP/dL",
        "Bound L",
        "Bound R"
        ])

    self.exploit_initialised = 0

    self.stats_hist = extract_initial_stats(
      self.dfs_primed[self.target],
      self.initial_boost
    )

    print(self.target)


  def new_price(self, date: int):

    print("new_price()")

    print("self.stats_hist:")
    print(self.stats_hist)

    if (date == 101):

      self.df_30.loc[100] = self.stats_hist

    else:

      print(self.df_30)

      print("date - 1: ", date - 1)

      print("cost: ", self.dfs_primed[self.target].iloc[-1]["Cost"])

      print("prices: ", self.engine.exposed_get_price(date - 1))

      new_row = update_history(
          self.df_30,
          date - 1,
          self.dfs_primed[self.target].iloc[-1]["Cost"],
          self.engine.exposed_get_price(date - 1),
          self.stats_hist
          )

      print("generated new_row: ", new_row)

      self.df_30.loc[date - 1] = new_row

      print(self.df_30)

      ##  print(self.df_30.to_csv())

      self.fallback_lp = 0

    ###########################################

    ##  Extracted from Optimum_chaser_MK3(). ##

    ###########################################

    if (date == 101):

      ##  If first date (!), then generate a price from historical stats.

      return self.stats_hist["Leader's Price"] + self.stats_hist["Leader's Price dL/dD"]

    elif (date > 101 and date < self.explore_exploit_boundary):

      print("explore exploit boundary:")
      print(self.explore_exploit_boundary)

      ##  Else traverse curve.

      self.df_30, new_date_boundary, lp = explore_prices(
          self.df_30,
          date
          )

      if (new_date_boundary != 0):

        ##  If the traversal has reached negative profit
        ##  early in the explore phase, end explore phase early.

        self.explore_exploit_boundary = new_date_boundary

      return lp_limited(lp, self.upper_limit)

    else:

      ##   Else if in explore phase...

      if date <= self.explore_exploit_boundary and self.exploit_initialised == 0:

        ##  Make a tiny increment on best known price so far, to deduce
        ##  which side of curve optimum is on, via positive or negative
        ##  gradient.

        self.exploit_initialised = 1

        lp = exploit_initialise_part_1(
            self.df_30
            )

        return lp_limited(lp, self.upper_limit)

      elif self.exploit_initialised == 1:

        ##  Calculate initial gradient, to deduce side of curve.

        self.exploit_initialised = 2

        self.df_orientation = exploit_initialise_part_2(
            self.df_30,
            self.df_orientation,
            date
        )

      if self.exploit_initialised != 0:

        # try:

          ##  Increment binary search for best gradient step, store result,
          ##  generate price.

          self.df_grads, lp = exploit_prices_gradient_gen(
              self.df_30,
              self.df_orientation,
              self.df_grads,
              date,
              self.explore_exploit_boundary
          )

          return lp_limited(lp, self.upper_limit)

        # except:

        #   return self.fallback_lp


class Optimum_chaser_MK1(Optimum_chaser_generic, Leader):

  ##  MK1 leader price can go very negative, during exploration.
  ##  This means it must quit early if negative profit hit.

    def __init__(self, name, engine):

        self.target = "Follower_Mk1"

        self.upper_limit = 2**63

        self.explore_exploit_boundary = 116

        ##  Set initial dL/dD value; this is a hyperparameter.

        self.initial_boost = 5

        Optimum_chaser_generic.__init__(self, name, engine)
        Leader.__init__(self, name, engine)

    def new_price(self, date: int):

        print("new_price() MK1")

        return Optimum_chaser_generic.new_price(self, date)


class Optimum_chaser_MK2(Optimum_chaser_generic, Leader):

    def __init__(self, name, engine):

        self.target = "Follower_Mk2"

        self.upper_limit = 2**63

        self.explore_exploit_boundary = 116

        ##  Set initial dL/dD value; this is a hyperparameter.

        self.initial_boost = 5

        Optimum_chaser_generic.__init__(self, name, engine)
        Leader.__init__(self, name, engine)

    def new_price(self, date: int):

        print("new_price() MK2")

        return Optimum_chaser_generic.new_price(self, date)


class Optimum_chaser_MK3(Optimum_chaser_generic, Leader):

  ##  MK3 leader price has an upper limit (same for MK6?).

    def __init__(self, name, engine):

        self.target = "Follower_Mk3"

        self.upper_limit = 15

        self.explore_exploit_boundary = 116

        ##  Set initial dL/dD value; this is a hyperparameter.

        self.initial_boost = 5

        Optimum_chaser_generic.__init__(self, name, engine)
        Leader.__init__(self, name, engine)

    def new_price(self, date: int):

        print("new_price() MK3")

        return Optimum_chaser_generic.new_price(self, date)


In [190]:
def lp_limited(lp, upper_limit):

  if lp > 1 and lp < upper_limit:

    return lp

  elif lp <= 1:

    return 1.1

  elif lp > upper_limit:

    return 14.9

In [191]:
def explore_prices(df: pd.DataFrame,
                   date: int) -> tuple[pd.DataFrame, float]:

  print(f"explore_prices(), {date}")

  ##  Explore; seek maximums along fluctuating curve.
  ##  Note that this sequence constantly grows leader's price.

  if (date <= 103):

    return df, 0, df.loc[date - 1]["Leader's Price"] + (df.loc[date - 1]["Leader's Price dL/dD"] * 2)

  elif df.loc[date - 1]["Profit"] < 0:

    ##  Escape hatch here in case profitability becomes negative.

    ##  Date boundary will need updating.

    print("found date boundary:")
    print(date - 1)

    return df, date - 1,  df.loc[date - 1]["Leader's Price"] + (df.loc[date - 1]["Leader's Price dL/dD"] * 2)

  else:

    ##  If 2nd derivative of profitability is higher than previously, and positive (curve on steepening incline)...

    if (df.loc[date - 1]["Profit d2P/dL2"] > df.loc[date - 2]["Profit d2P/dL2"] and df.loc[date - 1]["Profit d2P/dL2"] > 0):

      ##  Double rate of change of leader's price...

      return df, 0, df.loc[date - 1]["Leader's Price"] + (df.loc[date - 1]["Leader's Price dL/dD"] + df.loc[date - 1]["Leader's Price d2L/dD2"] * 2)

    elif df.loc[date - 1]["Profit d2P/dL2"] < df.loc[date - 2]["Profit d2P/dL2"] and df.loc[date - 1]["Profit d2P/dL2"] > 0:

      ##  2nd derivative is decreasing (curve on flattening incline, slow down whilst passing optimum)...

      ##  Half rate of change of leader's price...

      return df, 0, df.loc[date - 1]["Leader's Price"] + (df.loc[date - 1]["Leader's Price dL/dD"] + df.loc[date - 1]["Leader's Price d2L/dD2"] / 2)

    else:

      ##  2nd derivative is negative (curve on decline)...

      ##  Double rate of change of leader's price...

      return df, 0, df.loc[date - 1]["Leader's Price"] + (df.loc[date - 1]["Leader's Price dL/dD"] + df.loc[date - 1]["Leader's Price d2L/dD2"] * 2)


In [192]:
def exploit_initialise_part_1(df_30: pd.DataFrame) -> tuple[float]:

  print("exploit_init_part_1()")

  ##  Exploit initialisation.

  ##  This function finds the most profitable LP value, and adds a small increment.

  most_profitable = df_30.loc[df_30["Profit"].idxmax()]

  lp = most_profitable["Leader's Price"]

  lp_inc = lp + 0.01

  return lp_inc

In [193]:
def exploit_initialise_part_2(df_30: pd.DataFrame,
                              df_orientation: pd.DataFrame,
                              date: int) -> tuple[pd.DataFrame, float]:

  print("exploit_init_part_2()")

  ##  Exploit; seek optimum of best known maximum on curve.
  ##  Find most profitable leader's price known so far.

  ##  This function finds dP/dL based on the tiny increment in price, and stores it.

  ##  The dataframe is expected to contain only 1 row, consisting solely of constants.

  new_df_row = {}

  ##  Take 2 rows with largest profit value (should contain LP and LP + 0.1).
  ##  Ensure sorted by date (index column).

  profitable_top_two = df_30.nlargest(2, "Profit").sort_index()

  print("profitable_top_two:")

  print(profitable_top_two)

  print("new_df_row:")

  print(new_df_row)

  new_df_row["Leader's Price"] = profitable_top_two["Leader's Price"].iloc[0]

  print(new_df_row)

  new_df_row["Profit"] = profitable_top_two["Profit"].iloc[0]

  print(new_df_row)

  new_df_row["Big dP/dL"] = profitable_top_two["Profit dP/dL"].iloc[0]

  print(new_df_row)

  ##  Calculate differences between profit values, select lowest row.

  new_df_row["Tiny dP/dL"] = profitable_top_two["Profit"].diff().fillna(0).iloc[-1]

  print(new_df_row)

  df_orientation.loc[date - 1] = new_df_row

  print("df_orientation:")

  print(df_orientation)

  return df_orientation

In [194]:
def exploit_prices_gradient_gen(df_30: pd.DataFrame,
                                df_orientation: pd.DataFrame,
                                df_grads: pd.DataFrame,
                                date: int,
                                date_boundary: int
                                ##  dl_dp_hyperparam: float
                                ) -> tuple[pd.DataFrame, float]:

  print("exploit_prices_gradient_gen()")

  ##  Exploit; seek optimum of best known maximum on curve.
  ##  Find most profitable leader's price known so far.

  ##  This function takes the previous gradient, and generates a new gradient.

  ##  1. Finds most profitable LP value.

  new_df_row = {}

  if df_grads.empty:

    print("Initialising first two gradients.")

    ##  2.a. Setup bounds for binary search.

    ##  Take the point when the gradient is 0, and the point
    ##  when the gradient is half the next point in df_30.

    most_profitable = df_30.loc[:date_boundary].loc[df_30["Profit"].idxmax()]

    print("most_profitable:")
    print(most_profitable)

    df_grads.loc[df_30["Profit"].idxmax()] = {
        "Leader's Price": most_profitable["Leader's Price"],
        "dP/dL": 0,
        "Bound L": 1,
        "Bound R": 0
    }

    print("df_grads:")
    print(df_grads)

    new_df_row = {
        "Leader's Price": most_profitable["Leader's Price"],
        "dP/dL": most_profitable["Leader's Price dL/dD"] / 2, ##
        "Bound L": 0,
        "Bound R": 1
    }

    print("new_df_row:")
    print(new_df_row)

    new_lp = most_profitable["Leader's Price"] + most_profitable["Leader's Price dL/dD"] / 2

  else:

    ##  2.b. Post-initialisation.

    ##  Run a binary search to generate new gradient, based on past gradients generated.

    print("df_grads pre update_bounds():")
    print(df_grads)

    df_grads = update_bounds(df_30, df_grads)

    print("df_grads post update_bounds():")
    print(df_grads)

    new_df_row = {
        "Leader's Price": df_grads["Leader's Price"].iloc[0],
        "dP/dL": gen_midpoint_grad(df_grads),
        "Bound L": 0,
        "Bound R": 0
    }

    print("new_df_row:")
    print(new_df_row)

  df_grads.loc[date - 1] = new_df_row

  print("df_grads size 2+:")
  print(df_grads)

  print("Tiny head 1:")
  print(df_orientation["Tiny dP/dL"].iloc[0])

  if df_orientation["Tiny dP/dL"].iloc[0] > 0:

    new_lp = new_df_row["Leader's Price"] + new_df_row["dP/dL"]

  else:

    new_lp = new_df_row["Leader's Price"] + (new_df_row["dP/dL"] * -1)

  print("new_lp:")
  print(new_lp)

  return df_grads, new_lp

In [195]:
def update_bounds(df_30: pd.DataFrame, df_grads: pd.DataFrame):

  print("update_bounds()")

  ##  Take the last row in dataframe, extract gradient.

  print(df_grads.iloc[-1])

  last_grad_added = df_grads.iloc[-1]["dP/dL"]

  print("last_grad_added:")
  print(last_grad_added)

  ##  Duplicate dataframe.

  df_grads_addendum = df_grads.copy()

  ##  Find absolute difference between extracted gradient and all others.

  df_grads_addendum["dP/dL diff abs"] = (
      df_grads["dP/dL"] - last_grad_added
  ).abs()

  print("df_grads_addendum:")
  print(df_grads_addendum)

  ##  Sort by gradient.

  df_grads_diffs = df_grads_addendum.sort_values(by = "dP/dL diff abs")

  print("df_grads_diffs:")
  print(df_grads_diffs)

  print("df_grads_diffs dP/dL < last_grad_added:")
  print(df_grads_diffs[
      df_grads_diffs["dP/dL"] < last_grad_added
      ])

  print("df_grads_diffs dP/dL >= last_grad_added:")
  print(df_grads_diffs[
      df_grads_diffs["dP/dL"] >= last_grad_added
      ])

  ##  Edge case that could be handled here: df_grads features only 2 gradients, both same.

  boundaries = {
      "L":
      df_grads_diffs[
      df_grads_diffs["dP/dL"] < last_grad_added
      ].iloc[0],
      "R":
      df_grads_diffs[
      df_grads_diffs["dP/dL"] >= last_grad_added
      ].iloc[0]
  }

  boundaries_date = {
      "L":
      df_grads_diffs[
      df_grads_diffs["dP/dL"] < last_grad_added
      ].index[0],
      "R":
      df_grads_diffs[
      df_grads_diffs["dP/dL"] >= last_grad_added
      ].index[0]
  }

  print("boundaries:")
  print(boundaries)

  ##  If more than 2 boundaries stored, deduce which 2 are best.

  if len(df_grads) > 2:

    ##  Wipe previously stored boundaries.

    df_grads["Bound L"] = 0

    df_grads["Bound R"] = 0

    ##  Compare distances of latest gradient added to closest boundaries.
    ##  Choose closest two.

    ##  This is done by taking the date of each boundary,
    ##  and looking up the profitability.

    profit_l = df_30.loc[boundaries_date["L"]]["Profit"]

    profit_r = df_30.loc[boundaries_date["R"]]["Profit"]

    print("(profit_l, profit_r):")
    print((profit_l, profit_r))

    if profit_l < profit_r:

      ##  Set boundaries to R and centre.

      print("test1")

      new_l_loc = df_grads.index[-1]

      print("new_l_loc:")
      print(new_l_loc)

      new_r_loc = df_grads.index[df_grads["dP/dL"] == boundaries["R"]["dP/dL"]][0]

      print("new_r_loc:")
      print(new_r_loc)

    elif profit_l > profit_r:

      ##  Set boundaries to L and centre.

      print("test2")

      new_l_loc = df_grads.index[df_grads["dP/dL"] == boundaries["L"]["dP/dL"]][0]

      print("new_l_loc:")
      print(new_l_loc)

      new_r_loc = df_grads.index[-1]

      print("new_r_loc:")
      print(new_r_loc)

    df_grads.at[new_l_loc, "Bound L"] = 1

    df_grads.at[new_r_loc, "Bound R"] = 1

    print("df_grads:")
    print(df_grads)

  return df_grads

In [196]:
def gen_midpoint_grad(df_grads: pd.DataFrame):

  print("gen_midpoint_grad()")

  print("df_grads L matches")

  print(df_grads["Bound L"].map(lambda x: x == 1))

  l_bound = df_grads[df_grads["Bound L"].map(lambda x: x == 1)].index[0]

  r_bound = df_grads[df_grads["Bound R"].map(lambda x: x == 1)].index[0]

  print("l_bound")
  print(l_bound)

  print("r_bound")
  print(r_bound)

  diff = (df_grads.loc[r_bound]["dP/dL"] - df_grads.loc[l_bound]["dP/dL"])

  midpoint = df_grads.loc[l_bound]["dP/dL"] + diff / 2

  return midpoint

Please implement your leaders below. Please clearly document:
*   State which leader(s) is designed to play against each follower.

# Simulation

Below is the GUI interface. Please select a leader from the dropdown menu and a follower from the dropdown menu, then click “Connect.” Once the status updates to “Connected to *your_selected_leader* and *your_selected_follower*” click “Start Simulation” to begin. If you wish to save the generated data, click “Export Data.” The dataset will be saved in the “run” folder.

In [271]:
display(Javascript('''google.colab.output.setIframeHeight(0, true, {maxHeight: 5000})'''))
engine = Engine()
app = GUI(engine, Leader, group_num)

<IPython.core.display.Javascript object>

1.8071091994472521
Follower_Mk1


In [198]:
# leader = Optimum_chaser_MK3("test_mk1", engine)
# new_row = update_history(
#     leader.df_30,
#     101,
#     leader.dfs_primed["Follower_Mk1"].iloc[-1]["Cost"],
#     (leader.new_price(101), 20),
#     leader.stats_hist
#     )
# filtered_dict = {k: float(v) for k, v in new_row.items() if k in leader.df_30.columns}
# leader.df_30.loc[101] = new_row

**Initial data exploration**

In [199]:
# from IPython.display import SVG, display
# # Replace with your Raw GitHub URL
# url = "https://raw.githubusercontent.com/devicons/devicon/master/icons/python/python-original.svg"
# display(SVG(url=url))